# Sprints AI-Powered Learner Success & Support Agent
## Notebook 02: PyTorch Machine Learning & Deep Sequence Modeling (Sprints 2 & 3)
**Client:** Sprints (sprints.ai)  
**Track:** AI Pioneers Nationwide Cohort  
**Author:** Mohammed Wael (Lead AI Engineer)

---

### Notebook Overview
This notebook integrates **Sprint 2 (Intro to Machine Learning)** and **Sprint 3 (Deep Learning)**:
1. **PyTorch Supervised Risk Classifier (Sprint 2)**:
   - Feed-forward Multi-Layer Perceptron (MLP) with BatchNorm and Dropout.
   - Classifies learners as High Risk vs Low Risk.
   - Meets acceptance criteria: **F1 score $\ge$ 0.82 on the test set**, minimizing false negatives.
2. **PyTorch LSTM Sequence Model (Sprint 3)**:
   - Deep recurrent architecture processing $[q_1, q_2, q_3, q_4, q_5]$ sequential quiz vectors.
   - Theoretical and empirical analysis of the **forget gate** discarding early noise and combating the vanishing gradient problem.
   - Predicts final capstone project outcome.
3. **Latent Space Projections**:
   - Extracting penultimate layer embeddings and projecting via PCA / t-SNE to validate behavioral cluster separation.

In [1]:
import os
import sys
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, r2_score
from sklearn.decomposition import PCA

# Add scripts directory
sys.path.append(os.path.join('..', 'scripts'))
from train_pytorch import load_and_preprocess_data, LearnerRiskClassifier, LearnerSequenceLSTM

print(f"PyTorch Version: {torch.__version__}")

PyTorch Version: 2.14.1+cpu


### 1. Data Ingestion & Feature Engineering

In [2]:
csv_path = os.path.join('..', 'data', 'sprints_learner_data.csv')
df = load_and_preprocess_data(csv_path)
print(f"Processed DataFrame shape: {df.shape}")

Loaded dataset from ..\data\sprints_learner_data.csv: 2000 rows, 10 columns.
Missing values per quiz before imputation:
 week_1_quiz     15
week_2_quiz    109
week_3_quiz    147
week_4_quiz    262
week_5_quiz    366
dtype: int64
Imputation and feature engineering complete. No NaNs remaining.
Processed DataFrame shape: (2000, 16)


### 2. Sprint 2: PyTorch Supervised Feed-Forward Neural Network (Risk Classifier)
We evaluate the saved checkpoint `models/risk_classifier.pth` on the test set.

In [3]:
ckpt_path = os.path.join('..', 'models', 'risk_classifier.pth')
ckpt = torch.load(ckpt_path, weights_only=False)

feature_cols = ckpt['feature_cols']
print("Model Input Features:", feature_cols)

X = df[feature_cols].values
y = df['dropped_out'].values

# Split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

X_test_scaled = (X_test - ckpt['scaler_mean']) / ckpt['scaler_scale']

model = LearnerRiskClassifier(input_dim=len(feature_cols), hidden_dim=64)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

with torch.no_grad():
    logits = model(torch.tensor(X_test_scaled, dtype=torch.float32))
    probs = torch.sigmoid(logits).squeeze().numpy()

threshold = ckpt.get('decision_threshold', 0.42)
preds = (probs >= threshold).astype(int)

print("\n" + "="*50)
print("TEST SET CLASSIFICATION REPORT:")
print(classification_report(y_test, preds, target_names=['Low Risk', 'High Risk']))
print("ROC-AUC Score:", round(roc_auc_score(y_test, probs), 4))
print("="*50)

Model Input Features: ['week_1_quiz', 'week_2_quiz', 'week_3_quiz', 'week_4_quiz', 'week_5_quiz', 'score_delta_w1_w5', 'quiz_mean', 'quiz_std', 'quiz_min', 'recent_performance_ratio', 'missing_quiz_count']

TEST SET CLASSIFICATION REPORT:
              precision    recall  f1-score   support

    Low Risk       1.00      1.00      1.00       307
   High Risk       1.00      1.00      1.00        93

    accuracy                           1.00       400
   macro avg       1.00      1.00      1.00       400
weighted avg       1.00      1.00      1.00       400

ROC-AUC Score: 1.0


#### Confusion Matrix & Latent Space Clustering

In [4]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0], xticklabels=['Low Risk', 'High Risk'], yticklabels=['Low Risk', 'High Risk'])
axes[0].set_title('Confusion Matrix (Zero False Negatives)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

# Latent Space Projection (PCA)
with torch.no_grad():
    latent_reps = model.extract_latent_features(torch.tensor(X_test_scaled, dtype=torch.float32)).numpy()

pca = PCA(n_components=2)
latent_pca = pca.fit_transform(latent_reps)

sc = axes[1].scatter(latent_pca[:, 0], latent_pca[:, 1], c=y_test, cmap='coolwarm', alpha=0.8, edgecolors='k')
plt.colorbar(sc, ax=axes[1], label='Ground Truth Dropout Status')
axes[1].set_title('Latent Space Distribution (Behavioral Clusters)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Latent PC 1')
axes[1].set_ylabel('Latent PC 2')

plt.tight_layout()
plt.show()

C:\Users\mw851\AppData\Local\Temp\ipykernel_4980\3625286158.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 3. Sprint 3: PyTorch Deep Sequence LSTM Model
Mathematical formulation of the LSTM cell:
$$\begin{aligned}
f_t &= \sigma(W_f \cdot [h_{t-1}, x_t] + b_f) \quad &\text{(Forget Gate: discards irrelevant early noise)} \\
i_t &= \sigma(W_i \cdot [h_{t-1}, x_t] + b_i) \quad &\text{(Input Gate: selects new evidence to store)} \\
\tilde{C}_t &= \tanh(W_c \cdot [h_{t-1}, x_t] + b_c) \quad &\text{(Candidate Cell State Update)} \\
C_t &= f_t \odot C_{t-1} + i_t \odot \tilde{C}_t \quad &\text{(Cell State: additive constant error carousel)} \\
o_t &= \sigma(W_o \cdot [h_{t-1}, x_t] + b_o) \quad &\text{(Output Gate)} \\
h_t &= o_t \odot \tanh(C_t) \quad &\text{(Hidden State)}
\end{aligned}$$

The forget gate $f_t$ allows the network to disregard early-week anomalies (e.g., student illness or setup issues in Week 1) while retaining cumulative mastery trends.

In [5]:
lstm_ckpt_path = os.path.join('..', 'models', 'sequence_predictor.pth')
ckpt_lstm = torch.load(lstm_ckpt_path, weights_only=False)

lstm_model = LearnerSequenceLSTM(input_size=1, hidden_size=ckpt_lstm['hidden_size'], num_layers=ckpt_lstm['num_layers'])
lstm_model.load_state_dict(ckpt_lstm['model_state_dict'])
lstm_model.eval()

seq_cols = ['week_1_quiz', 'week_2_quiz', 'week_3_quiz', 'week_4_quiz', 'week_5_quiz']
seqs = df[seq_cols].values / 100.0
targets = df['project_score'].values

X_tr_s, X_te_s, y_tr_s, y_te_s = train_test_split(seqs, targets, test_size=0.2, random_state=42)

with torch.no_grad():
    tensor_seq = torch.tensor(X_te_s, dtype=torch.float32).unsqueeze(-1)
    preds_norm, _ = lstm_model(tensor_seq)
    predicted_projects = preds_norm.squeeze().numpy() * 100.0

r2 = r2_score(y_te_s, predicted_projects)
rmse = np.sqrt(np.mean((y_te_s - predicted_projects)**2))

print(f"Sequence LSTM Test RMSE: {rmse:.2f} points")
print(f"Sequence LSTM Test R^2 Score: {r2:.4f}")

Sequence LSTM Test RMSE: 7.80 points
Sequence LSTM Test R^2 Score: 0.8841


In [6]:
# Plotting actual vs predicted trajectories
plt.figure(figsize=(10, 5))
weeks = [1, 2, 3, 4, 5]
sample_idx = [3, 8, 15, 22]

for idx in sample_idx:
    p = plt.plot(weeks, X_te_s[idx] * 100.0, marker='o', label=f'Learner #{idx} Weekly Quizzes')
    col = p[0].get_color()
    plt.scatter(6, predicted_projects[idx], marker='*', s=150, color=col, label=f'Pred Capstone: {predicted_projects[idx]:.1f}')
    plt.scatter(6, y_te_s[idx], marker='x', s=100, color=col)

plt.title('PyTorch LSTM Sequential Grade Trajectory & Capstone Forecast', fontsize=13, fontweight='bold')
plt.xlabel('Bootcamp Timeline')
plt.ylabel('Assessment Score')
plt.xticks([1, 2, 3, 4, 5, 6], ['W1 Quiz', 'W2 Quiz', 'W3 Quiz', 'W4 Quiz', 'W5 Quiz', 'Project Score'])
plt.legend(bbox_to_anchor=(1.04, 1), loc="upper left", fontsize=8)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

C:\Users\mw851\AppData\Local\Temp\ipykernel_4980\3685502269.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
